## Freeze the BGE representation-dependent parameters using clean reference data


MMD and Sinkhorn depend on the geometry of the representation space. When switching from MiniLM to BGE, the distances between points change, so those parameters must be recalculated for BGE using only the 700 clean reference campaigns, never using defective ones. That is exactly what was done with MiniLM for MMD: generate embeddings for the 700 clean reference campaigns and applied `median_heuristic_sigma(...)`, obtaining the fixed sigma for MiniLM.

In [1]:
import importlib,json,numpy as np,pandas as pd
from scipy.spatial.distance import pdist
from sentence_transformers import SentenceTransformer

import src.paths as paths,src.metrics as metrics,src.representations as representations
importlib.reload(paths);importlib.reload(metrics);importlib.reload(representations)

from src.paths import get_project_paths
from src.metrics import median_heuristic_sigma
from src.representations import serialize_campaign_dataset,encode_serialized_campaigns

PATHS=get_project_paths()
DATA_DIR=PATHS["processed"]
BGE_DIR=DATA_DIR/"robustness_bge"

In [2]:
# Load frozen BGE config
with open(BGE_DIR/"entity_bge_configuration.json") as f:
    bge_config=json.load(f)

BGE_MODEL_NAME=bge_config["model"]
BGE_MODEL_REVISION=bge_config["revision"]
BGE_BATCH_SIZE=bge_config["batch_size"]

bge_model=SentenceTransformer(BGE_MODEL_NAME,revision=BGE_MODEL_REVISION)

print(BGE_MODEL_NAME)
print(BGE_MODEL_REVISION)
print(bge_model.get_sentence_embedding_dimension())

BAAI/bge-base-en-v1.5
a5beb1e3e68b9ab74eb54cfd186867f64f240e1a
768


In [3]:
# Load 700 clean reference campaigns
clean_campaigns=pd.read_csv(DATA_DIR/"clean_structured_campaigns.csv",parse_dates=["timestamp"])
split_df=pd.read_csv(DATA_DIR/"structural_campaign_split.csv")

train_ids=set(split_df.loc[split_df["split"]=="train","campaign_id"])

clean_reference=(
    clean_campaigns[
        clean_campaigns["campaign_id"].isin(train_ids)
    ]
    .copy()
    .reset_index(drop=True)
)

assert clean_reference["campaign_id"].nunique()==700

print("Reference campaigns:",clean_reference["campaign_id"].nunique())

Reference campaigns: 700


In [4]:
# Serialize
clean_reference_serialized=serialize_campaign_dataset(clean_reference)

assert clean_reference_serialized.shape==(700,2)

print("Serialized reference:",clean_reference_serialized.shape)

Serialized reference: (700, 2)


In [5]:
# Generate clean reference BGE embeddings
bge_reference_embeddings,runtime=encode_serialized_campaigns(
    clean_reference_serialized,
    bge_model,
    batch_size=BGE_BATCH_SIZE
)

assert bge_reference_embeddings.shape==(700,bge_model.get_sentence_embedding_dimension())
assert np.isfinite(bge_reference_embeddings).all()
assert np.allclose(np.linalg.norm(bge_reference_embeddings,axis=1),1.0,atol=1e-5)

print("Reference embedding shape:",bge_reference_embeddings.shape)
print("Runtime:",round(runtime,2),"s")

Campaigns: 700
Chunks to encode: 700


Batches:   0%|          | 0/22 [00:00<?, ?it/s]

Reference embedding shape: (700, 768)
Runtime: 501.1 s


In [6]:
# Freeze MMD sigma
BGE_MMD_SIGMA=median_heuristic_sigma(bge_reference_embeddings)

print("Frozen BGE MMD sigma:",BGE_MMD_SIGMA)

Frozen BGE MMD sigma: 0.3873605130049743


In [7]:
# Freeze Sinkhorn scale and blur
BGE_SINKHORN_MEDIAN_SCALE=float(np.median(pdist(bge_reference_embeddings,metric="euclidean")))
BGE_SINKHORN_BLUR=0.10*BGE_SINKHORN_MEDIAN_SCALE

print("BGE Sinkhorn median scale:",BGE_SINKHORN_MEDIAN_SCALE)
print("Frozen BGE Sinkhorn blur:",BGE_SINKHORN_BLUR)

BGE Sinkhorn median scale: 0.3873605130049747
Frozen BGE Sinkhorn blur: 0.038736051300497475


In [8]:
# Save frozen artifacts
np.save(
    BGE_DIR/"campaign_bge_reference_clean.npy",
    bge_reference_embeddings
)

clean_reference_serialized[["campaign_id"]].to_csv(
    BGE_DIR/"campaign_bge_reference_clean_ids.csv",
    index=False
)

bge_scale_config={
    "model":BGE_MODEL_NAME,
    "revision":BGE_MODEL_REVISION,
    "reference_campaigns":700,
    "embedding_dimension":int(bge_model.get_sentence_embedding_dimension()),
    "normalized":True,
    "mmd_sigma_rule":"clean-reference median heuristic",
    "mmd_sigma":float(BGE_MMD_SIGMA),
    "sinkhorn_scale_rule":"median pairwise Euclidean distance on clean reference embeddings",
    "sinkhorn_median_scale":float(BGE_SINKHORN_MEDIAN_SCALE),
    "sinkhorn_blur_rule":"0.10 * clean-reference median pairwise Euclidean distance",
    "sinkhorn_blur":float(BGE_SINKHORN_BLUR)
}

with open(BGE_DIR/"bge_metric_scales.json","w") as f:
    json.dump(bge_scale_config,f,indent=4)

print("BGE metric scales frozen.")

BGE metric scales frozen.
